# Generate Dashboard

Runs `export_dashboard_data.py` + `build_dashboard.py` to refresh
`dashboard/river3_scouting_report.html`. The dashboard itself is interactive (game window,
role, reference tier, and cluster count are all filterable in the browser), so this export
step just needs to run once per data refresh -- it computes everything for every
role/tier/k combination up front. Run `00_pull_data.ipynb` first if you want the dashboard
to include newly-fetched games.

Publishing the result as a Claude Artifact has to happen in a Claude Code conversation
(ask Claude to publish `dashboard/river3_scouting_report.html`) -- there's no notebook-side
publish step.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('../src'))
os.chdir('..')

import export_dashboard_data
import build_dashboard

RIOT_ID = 'river3#na1'

In [ ]:
sys.argv = ['export_dashboard_data.py', '--riot-id', RIOT_ID]
export_dashboard_data.run(export_dashboard_data.parse_args())

In [ ]:
sys.argv = ['build_dashboard.py']
build_dashboard.run(build_dashboard.parse_args())

## Quick sanity check

Peek at the payload that just got embedded into the dashboard, without opening the HTML.

In [ ]:
import json

with open('data/processed/dashboard_data.json', encoding='utf-8') as f:
    data = json.load(f)

print('Record:', data['record'])
print('Role counts:', data['role_counts'])
print('Champion pool:', data['champion_counts'])
print('Reference tiers available:', data['tiers'])
print('Cluster-count options:', data['k_options'])
print()
for role, role_data in data['roles'].items():
    if data['role_counts'].get(role):
        summary = role_data['clusters_by_k']['4']['summary']
        print(f"{role}:")
        for c in sorted(summary, key=lambda c: c['win_rate']):
            print(f"  {c['name']:<22} win_rate={c['win_rate']:.1%}  n={c['n']}")